# End-to-End Pipeline: Probabilistic Nowcasting with Retrieval-Augmented Diffusion

**Skripsi**: Probabilistic Nowcasting Hujan Ekstrem Pemicu Banjir Bandang Sitaro

Notebook ini merangkum **seluruh pipeline** dari data mentah hingga evaluasi model, memastikan konsistensi dengan script `src/train.py` dan `src/evaluate_full.py`.

### Tahapan Pipeline:
1.  **Data Loading & Splitting**: Memuat data ERA5 dan membaginya secara **Temporal** (Train/Val/Test) untuk mencegah data leakage.
2.  **Graph Construction**: Membangun graph spasial (Node Siau, Tagulandang, Biaro).
3.  **Feature Engineering**: Normalisasi dan sliding window sequence.
4.  **Retrieval Indexing**: Membangun database FAISS hanya dari data **Training**.
5.  **Model Initialization**: Menyiapkan Spatio-Temporal GNN dan Conditional Diffusion Model.
6.  **Training**: Loop pelatihan dengan validasi loss.
7.  **Evaluation**: Load checkpoint terbaik dan evaluasi di Test Set (CRPS, RMSE).

---

In [ ]:
import sys
import os
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader

# Tambahkan root project ke path
sys.path.append(os.path.abspath('..'))

# Import modul custom
from src.data.temporal_loader import TemporalGraphDataset, collate_temporal_graphs
from src.models.gnn import SpatioTemporalGNN
from src.models.diffusion import RainForecaster
from src.retrieval.base import RetrievalDatabase
from src.train import train_one_epoch, validate

# Config GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## 1. Data Loading & Temporal Split
Kita membagi data secara ketat berdasarkan waktu:
*   **Train**: 2005 - 2018 (Belajar pola jangka panjang)
*   **Val**: 2019 - 2021 (Tuning hyperparameter & Early stopping)
*   **Test**: 2022 - 2025 (Evaluasi performa 'masa depan')

In [ ]:
data_path = '../data/raw/sitaro_era5_2005_2025.parquet'
df = pd.read_parquet(data_path)
df['date'] = pd.to_datetime(df['date'])

# Temporal Split
train_df = df[df['date'].dt.year <= 2018].copy()
val_df = df[(df['date'].dt.year >= 2019) & (df['date'].dt.year <= 2021)].copy()
test_df = df[df['date'].dt.year >= 2022].copy()

print(f"Train samples: {len(train_df)}")
print(f"Val samples:   {len(val_df)}")
print(f"Test samples:  {len(test_df)}")

## 2. Dataset & Normalization
Penting: Statistik normalisasi (Mean/Std) **hanya dihitung dari Training Data** dan diterapkan ke Val/Test. Ini mencegah informasi masa depan bocor ke proses training.

In [ ]:
# Inisialisasi Dataset
# Mode 'train' akan menghitung statistik normalisasi baru
train_dataset = TemporalGraphDataset(train_df, seq_len=6, mode='train')
stats = train_dataset.stats  # Simpan statistik ini!

# Mode 'test' menggunakan statistik yang sudah ada
val_dataset = TemporalGraphDataset(val_df, seq_len=6, mode='test', stats=stats)
test_dataset = TemporalGraphDataset(test_df, seq_len=6, mode='test', stats=stats)

print("Normalization Stats computed from Train Set:")
print(stats)

# DataLoader
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, collate_fn=collate_temporal_graphs)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, collate_fn=collate_temporal_graphs)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, collate_fn=collate_temporal_graphs)

## 3. Retrieval Indexing
Membangun database vektor dari fitur historis (data Training saja). Saat inferensi, model akan mencari 'hari yang mirip' dari database ini untuk membantu prediksi.

In [ ]:
retrieval_db = RetrievalDatabase(k=3)

# Ekstrak fitur dari Training data untuk database
# (Menggunakan kolom fitur saja, bukan target)
print("Building Retrieval Index...")
feature_cols = ['temperature_2m', 'relative_humidity_2m', 'surface_pressure', 'wind_speed_10m', 'wind_direction_10m']

# Kita ambil data per jam dari train_df
# Normalisasi fitur sebelum masuk index sangat penting
train_features = train_df[feature_cols].values
train_features_norm = (train_features - stats['c_mean'].numpy()) / (stats['c_std'].numpy() + 1e-5)

# Build index
retrieval_db.build_index(train_features_norm)
print(f"Index contains {retrieval_db.index.ntotal} vectors.")

## 4. Model Initialization
Kita menggunakan dua komponen utama:
1.  **Spatio-Temporal GNN:** Memproses urutan graph cuaca (6 jam terakhir) menjadi embedding.
2.  **Conditional Diffusion Model:** Men-generate prediksi hujan berdasarkan embedding GNN dan hasil Retrieval.
    *   *Note: Di sini Diffusion Model dibungkus dalam kelas `RainForecaster`.*

In [ ]:
# GNN (Encoder)
st_gnn = SpatioTemporalGNN(
    node_features=5, 
    hidden_dim=64, 
    output_dim=64
).to(device)

# Diffusion (Generative Forecaster)
forecaster = RainForecaster(
    context_dim=64,   # Output dari GNN
    retrieval_dim=5,  # Dimensi fitur retrieval
    hidden_dim=64
).to(device)

optimizer = torch.optim.Adam(list(st_gnn.parameters()) + list(forecaster.parameters()), lr=1e-3)

## 5. Training Loop
Melatih model end-to-end. Outputnya adalah loss gabungan (Noise Prediction Error).

In [ ]:
NUM_EPOCHS = 1  # Contoh 1 epoch saja untuk demonstrasi. Gunakan lebih banyak untuk hasil optimal.

print("Starting Training...")
for epoch in range(NUM_EPOCHS):
    # Train
    train_loss = train_one_epoch(st_gnn, forecaster, retrieval_db, train_loader, optimizer, device)
    
    # Validate
    val_loss = validate(st_gnn, forecaster, retrieval_db, val_loader, device)
    
    print(f"Epoch {epoch+1}/{NUM_EPOCHS} - Train Loss: {train_loss:.4f} - Val Loss: {val_loss:.4f}")
    
    # Save checkpoint dummy (for pipeline demo)
    # Di production, gunakan logic 'save best model' di src/train.py
    pass

## 6. Real Evaluation (Test Set)
Menggunakan checkpoint terbaik yang sudah disimpan (`models/diffusion_chkpt.pth`) untuk evaluasi metrik sebenarnya.

In [ ]:
from src.inference import load_model_and_stats, run_inference_real

CHECKPOINT_PATH = "../models/diffusion_chkpt.pth"

if os.path.exists(CHECKPOINT_PATH):
    # Load Best Model
    loaded_model, loaded_stats, loaded_db = load_model_and_stats(CHECKPOINT_PATH)
    loaded_model.to(device)
    
    print("\n--- Running Evaluation on Subset of Test Data ---")
    # Ambil 1 batch dari test_loader untuk sample
    batch = next(iter(test_loader))
    x, edge_index, batch_idx, y, retrieval_k = batch
    
    x = x.to(device)
    edge_index = edge_index.to(device)
    batch_idx = batch_idx.to(device)
    
    # Karena struktur batch GNN kompleks, kita gunakan run_inference_real untuk single points
    # atau implementasi evaluasi batch.
    # Di sini kita demonstrasikan load berhasil.
    print("Model checkpoint loaded successfully for evaluation.")
    
else:
    print("Checkpoint not found. Train usage example complete.")